<a href="https://colab.research.google.com/github/suelisena/smart_fraud_detector-/blob/main/C%C3%B3pia_de_2_2_Deteccao_de_Fraudes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 # 2.2. Detecção de Fraudes

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Bibliotecas

In [2]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from google.colab import files



### Importa e Visualiza

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Fraude_Detectetion/fraud_detection/fraudes.csv")
df.head()

,transaction_id,customer_id,amount,time,location,is_fraud
0,TRANS_1668,CUST_257,3871.09,10,Loja Física,0
1,TRANS_731,CUST_391,4222.56,23,Online,1
2,TRANS_1644,CUST_360,4451.62,20,Online,0
3,TRANS_499,CUST_342,2698.11,4,Online,0
4,TRANS_827,CUST_146,2305.39,2,Online,0


### Pré-Processamento

In [ ]:
# Remove colunas sem valor semântico (IDs)
df = df.drop(columns=['transaction_id', 'customer_id'], errors='ignore')

# === 3. Inserir mais dados de fraude simulados ===
np.random.seed(42)
fraudes_simuladas = pd.DataFrame({
    "amount": np.random.uniform(10000, 20000, 500),  # valores altos
    "time": np.random.randint(0, 24, 500),           # horas variadas
    "location": ["Online"] * 500,                    # sempre Online
    "is_fraud": [1] * 500                            # marcados como fraude
})

df = pd.concat([df, fraudes_simuladas], ignore_index=True)

# === 4. Pré-processamento ===
df_encoded = pd.get_dummies(df, columns=['location'], drop_first=True)

X = df_encoded.drop(columns=['is_fraud'])
y = df_encoded['is_fraud']

scaler = StandardScaler()
X[['amount', 'time']] = scaler.fit_transform(X[['amount', 'time']])

# === 5. Balanceamento com SMOTE ===
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# === 6. Divisão em treino e teste ===
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)
print(X_train.head())

        amount      time  location_Online
3705 -0.299192  0.808968             True
3857  3.259055 -1.650661             True
2672 -0.717712  0.672322             True
4386 -0.749293  0.945614            False
4381  0.022701 -0.557493             True


### Treinamento

In [ ]:
model = MLPClassifier(hidden_layer_sizes=(50, 30, 20), max_iter=1000,
                      random_state=42, learning_rate_init=0.01, activation='relu')
model.fit(X_train, y_train)



MLPClassifier(hidden_layer_sizes=(50, 30, 20), learning_rate_init=0.01,
              max_iter=1000, random_state=42)

### Avaliação

In [ ]:
y_pred = model.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.84      0.82      0.83       495
           1       0.81      0.84      0.83       462

    accuracy                           0.83       957
   macro avg       0.83      0.83      0.83       957
weighted avg       0.83      0.83      0.83       957



Download arquivos pkl

In [ ]:
# Salva os arquivos do modelo e do scaler
joblib.dump(model, "fraude_model.pkl")
joblib.dump(scaler, "scaler.pkl")

files.download("fraude_model.pkl")
files.download("scaler.pkl")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Novos Dados

In [ ]:
# Lista de novas transações
new_transactions = [
    {"amount": 3871.09, "time": 10, "location": "Loja Física"},
    {"amount": 12.56, "time": 12, "location": "Online"},
    {"amount": 4451.62, "time": 20, "location": "Online"},
    {"amount": 38.09, "time": 20, "location": "Loja Física"},
    {"amount": 4222.56, "time": 23, "location": "Online"},
]


# Cria DataFrame
df_new = pd.DataFrame(new_transactions)

# Transforma variavel categórica
df_new_encoded = pd.get_dummies(df_new, columns=['location'], drop_first=True)

# Normaliza
df_new_encoded[['amount', 'time']] = scaler.transform(df_new_encoded[['amount', 'time']])

# Previsão
predictions = model.predict(df_new_encoded)
probabilities = model.predict_proba(df_new_encoded)[:, 1]

# Adiciona os resultados ao DataFrame original
df_new["fraud_probability"] = probabilities
df_new["is_fraud_predicted"] = predictions

# Exibr previsões
print(df_new[["amount", "time", "location", "fraud_probability", "is_fraud_predicted"]])


    amount  time     location  fraud_probability  is_fraud_predicted
0  3871.09    10  Loja Física           0.003635                   0
1    12.56    12       Online           0.426263                   0
2  4451.62    20       Online           0.000017                   0
3    38.09    20  Loja Física           0.737962                   1
4  4222.56    23       Online           0.999814                   1
